In [2]:
import os
import pandas as pd

# 1. Mendapatkan path ke folder Desktop secara otomatis
folder_path = r'C:\protofolio\ecommerce_analytic\data bersih'
folder_path_raw_data = r'C:\protofolio\ecommerce_analytic\data mentah'


# 3. Membaca file menggunakan pandas


df_transactions = pd.read_csv(os.path.join(folder_path_raw_data, 'transactions.csv')).copy()

dim_products= pd.read_csv(os.path.join(folder_path, 'dim_products_clean.csv')).copy()
dim_cutomers= pd.read_csv(os.path.join(folder_path, 'dim_customers_clean.csv')).copy()
dim_campaigns= pd.read_csv(os.path.join(folder_path, 'dim_campaigns_clean.csv')).copy()



df_transactions.info()



<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    103127 non-null  int64  
 1   timestamp         103127 non-null  str    
 2   customer_id       103127 non-null  int64  
 3   product_id        92678 non-null   float64
 4   quantity          103127 non-null  int64  
 5   discount_applied  103127 non-null  float64
 6   gross_revenue     92678 non-null   float64
 7   campaign_id       103127 non-null  int64  
 8   refund_flag       103127 non-null  int64  
dtypes: float64(3), int64(5), str(1)
memory usage: 7.1 MB


In [3]:
df_transactions.isnull().sum()

transaction_id          0
timestamp               0
customer_id             0
product_id          10449
quantity                0
discount_applied        0
gross_revenue       10449
campaign_id             0
refund_flag             0
dtype: int64

In [4]:
#df_transactions['refund_flag'].unique()
print(df_transactions['refund_flag'].value_counts())
#gross data minus
jumlah_minus = (df_transactions['gross_revenue'] < 0).sum()
print(f"Ada minus: {jumlah_minus}")

refund_flag
0    100098
1      3029
Name: count, dtype: int64
Ada minus: 2704


In [5]:
# Melihat sampel data yang gross_revenue-nya < 0
negative_revenue = df_transactions[df_transactions['gross_revenue'] < 0]

# berapa banyak dari data negatif ini yang punya refund_flag == 1
print(negative_revenue['refund_flag'].value_counts())

# gross_revenue negatif yang bukan refund (refund_flag == 0)
data_aneh = df_transactions[(df_transactions['gross_revenue'] < 0) & (df_transactions['refund_flag'] == 0)]

print(f"Jumlah data negatif yang BUKAN refund: {len(data_aneh)}")


refund_flag
1    2704
Name: count, dtype: int64
Jumlah data negatif yang BUKAN refund: 0


In [6]:
# 3. create kolom transaction date

df_transactions['timestamp'] = pd.to_datetime(df_transactions['timestamp'])
df_transactions['transaction_date'] = df_transactions['timestamp'].dt.date

df_transactions.info()
#display(df_transactions_clean['transaction_date'])
#display(df_transactions_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    103127 non-null  int64         
 1   timestamp         103127 non-null  datetime64[us]
 2   customer_id       103127 non-null  int64         
 3   product_id        92678 non-null   float64       
 4   quantity          103127 non-null  int64         
 5   discount_applied  103127 non-null  float64       
 6   gross_revenue     92678 non-null   float64       
 7   campaign_id       103127 non-null  int64         
 8   refund_flag       103127 non-null  int64         
 9   transaction_date  103127 non-null  object        
dtypes: datetime64[us](1), float64(3), int64(5), object(1)
memory usage: 7.9+ MB


In [7]:
print(df_transactions['refund_flag'].value_counts())

refund_flag
0    100098
1      3029
Name: count, dtype: int64


TRANSACTION ID

In [8]:
#cek transaction id unik
#  Menghitung total baris (Count *)
total_baris = df_transactions['transaction_id'].count()

#  Menghitung total transaksi yang unik (Count Distinct)
total_transaksi_unik = df_transactions['transaction_id'].nunique()

print("Total Baris:", total_baris)
print("Total Transaksi Unik:", total_transaksi_unik)

Total Baris: 103127
Total Transaksi Unik: 103127


In [9]:
#cek duplicate transaction id
df_transactions['transaction_id'].duplicated().sum()


np.int64(0)

In [10]:
#cek total duplicate row 
df_transactions.duplicated().sum()


np.int64(0)

REVENUE

In [11]:
#revenue >=0
# Menghitung berapa banyak baris yang revenues-nya di bawah 0 (minus)
data_minus_revenue = ((df_transactions['refund_flag'] == 0) & 
                    df_transactions['gross_revenue'] < 0).sum()
print(f"Jumlah data dengan revenue minus: {data_minus_revenue}")


Jumlah data dengan revenue minus: 0


In [12]:
#net revenue table

# kondisi 1 Jika refund, paksa net_revenue jadi 0.0
df_transactions.loc[df_transactions['refund_flag'] == 1, 'net_revenue'] = 0.0

#  Kondisi 2 Jika NORMAL (bukan refund), hitung pakai rumus potongan diskon
df_transactions.loc[df_transactions['refund_flag'] == 0, 'net_revenue'] = (
    df_transactions['gross_revenue'] * (1 - df_transactions['discount_applied'])
).round(2) #pembulatan


df_transactions[['refund_flag', 'gross_revenue', 'discount_applied', 'net_revenue']].head(15)

,refund_flag,gross_revenue,discount_applied,net_revenue
0,0,43.74,0.00,43.74
1,0,174.78,0.00,174.78
2,0,40.61,0.00,40.61
3,0,68.76,0.15,58.45
4,0,14.64,0.00,14.64
5,0,169.74,0.00,169.74
6,0,42.48,0.00,42.48
7,0,36.95,0.00,36.95
8,0,125.59,0.00,125.59
9,0,48.72,0.15,41.41


In [13]:
df_transactions['net_revenue'].describe()

count    93003.000000
mean        89.299673
std         92.870356
min          0.000000
25%         31.970000
50%         64.260000
75%        113.290000
max       1858.320000
Name: net_revenue, dtype: float64

In [14]:
df_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    103127 non-null  int64         
 1   timestamp         103127 non-null  datetime64[us]
 2   customer_id       103127 non-null  int64         
 3   product_id        92678 non-null   float64       
 4   quantity          103127 non-null  int64         
 5   discount_applied  103127 non-null  float64       
 6   gross_revenue     92678 non-null   float64       
 7   campaign_id       103127 non-null  int64         
 8   refund_flag       103127 non-null  int64         
 9   transaction_date  103127 non-null  object        
 10  net_revenue       93003 non-null   float64       
dtypes: datetime64[us](1), float64(4), int64(5), object(1)
memory usage: 8.7+ MB


QTY

In [15]:
#qty >=0
data_qty_minus= (df_transactions['quantity'] <0).sum()
print(f"jlh data qty minus: {data_qty_minus}")



jlh data qty minus: 0


DISCOUNT

In [16]:
#discountNilai min harus >= 0 ,max tidak boleh lebih dari 1.0 (alias diskon 100%)
df_transactions['discount_applied'].describe()


count    103127.000000
mean          0.041379
std           0.060286
min           0.000000
25%           0.000000
50%           0.000000
75%           0.050000
max           0.200000
Name: discount_applied, dtype: float64

In [17]:
# cek kondisi Diskon  hanya cek yang refund_flag-nya 0 (TIDAK REFUND)
diskon_ngaco = df_transactions[
    (df_transactions['refund_flag'] == 0) & 
    (df_transactions['discount_applied'] > df_transactions['gross_revenue'])
]

print(f"Jumlah transaksi dengan diskon tidak masuk akal (di luar refund): {len(diskon_ngaco)}")

Jumlah transaksi dengan diskon tidak masuk akal (di luar refund): 0


DATE

In [18]:
df_transactions['timestamp']=pd.to_datetime(df_transactions['timestamp'])

df_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    103127 non-null  int64         
 1   timestamp         103127 non-null  datetime64[us]
 2   customer_id       103127 non-null  int64         
 3   product_id        92678 non-null   float64       
 4   quantity          103127 non-null  int64         
 5   discount_applied  103127 non-null  float64       
 6   gross_revenue     92678 non-null   float64       
 7   campaign_id       103127 non-null  int64         
 8   refund_flag       103127 non-null  int64         
 9   transaction_date  103127 non-null  object        
 10  net_revenue       93003 non-null   float64       
dtypes: datetime64[us](1), float64(4), int64(5), object(1)
memory usage: 8.7+ MB


In [19]:
df_transactions.head()

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag,transaction_date,net_revenue
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0,2021-12-27,43.74
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0,2023-06-06,174.78
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0,2023-08-31,40.61
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0,2022-06-26,58.45
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0,2023-07-26,14.64


In [20]:
# Cek rentang tanggal transaksi
print(f"Transaksi dimulai dari: {df_transactions['timestamp'].min()}")
print(f"Transaksi berakhir di: {df_transactions['timestamp'].max()}")

Transaksi dimulai dari: 2021-01-01 00:12:50
Transaksi berakhir di: 2023-12-31 22:37:32


In [21]:
#null date 
print(f"null date : {df_transactions['timestamp'].isna().sum()}")

null date : 0


REFRENCE

In [22]:
#cek refrensi product id di dim product
missing_product_id= ~ df_transactions['product_id'].isin(dim_products['product_id'])
print(f"missing prodduct id: {missing_product_id}")

missing prodduct id: 0         False
1         False
2         False
3         False
4         False
          ...  
103122    False
103123    False
103124    False
103125    False
103126    False
Name: product_id, Length: 103127, dtype: bool


In [23]:
#cek refrensi cutomers id di dim customers
missing_customers_id= ~df_transactions['customer_id'].isin(dim_cutomers['customer_id'])
print(f"missing prodduct id: {missing_customers_id}")


missing prodduct id: 0         False
1         False
2         False
3         False
4         False
          ...  
103122    False
103123    False
103124    False
103125    False
103126    False
Name: customer_id, Length: 103127, dtype: bool


In [24]:
#cek refrensi cutomers id di dim customers
missing_campaigns_id= ~df_transactions['campaign_id'].isin(dim_campaigns['campaign_id'])
print(f"missing prodduct id: {missing_campaigns_id}")

missing prodduct id: 0          True
1         False
2         False
3         False
4         False
          ...  
103122     True
103123     True
103124     True
103125    False
103126    False
Name: campaign_id, Length: 103127, dtype: bool


In [25]:
#cek revenue outlier ekstrem
df_transactions['net_revenue'].describe()


count    93003.000000
mean        89.299673
std         92.870356
min          0.000000
25%         31.970000
50%         64.260000
75%        113.290000
max       1858.320000
Name: net_revenue, dtype: float64

In [26]:
#cek revenue outlier ekstrem
df_transactions.sort_values('net_revenue',ascending=False).head(10)

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag,transaction_date,net_revenue
63771,63772,2023-11-27 16:53:44,72115,496.0,4,0.00,1858.32,0,0,2023-11-27,1858.32
89725,89726,2022-01-15 13:08:00,11800,496.0,4,0.00,1858.32,25,0,2022-01-15,1858.32
39714,39715,2021-04-29 20:59:11,67585,223.0,4,0.00,1597.44,27,0,2021-04-29,1597.44
57523,57524,2021-09-28 02:07:00,54135,496.0,3,0.00,1393.74,24,0,2021-09-28,1393.74
53552,53553,2023-07-22 19:59:22,31613,496.0,3,0.00,1393.74,12,0,2023-07-22,1393.74
8709,8710,2023-10-10 16:18:32,93869,496.0,4,0.15,1579.57,44,0,2023-10-10,1342.63
58001,58002,2021-11-24 18:57:18,21602,1185.0,4,0.00,1224.76,2,0,2021-11-24,1224.76
15135,15136,2023-10-26 19:04:38,34218,1590.0,4,0.00,1207.32,16,0,2023-10-26,1207.32
95975,95976,2021-05-09 17:34:42,30826,1590.0,4,0.00,1207.32,7,0,2021-05-09,1207.32
97179,97180,2022-05-22 13:25:59,40571,1590.0,4,0.00,1207.32,48,0,2022-05-22,1207.32


In [27]:
#10 transaksi terbesar

df_transactions.sort_values('net_revenue',ascending=False)[['transaction_id','net_revenue']].head(10)

,transaction_id,net_revenue
63771,63772,1858.32
89725,89726,1858.32
39714,39715,1597.44
57523,57524,1393.74
53552,53553,1393.74
8709,8710,1342.63
58001,58002,1224.76
15135,15136,1207.32
95975,95976,1207.32
97179,97180,1207.32


In [28]:
df_transactions.sort_values('net_revenue',ascending=False)[['transaction_id','quantity','gross_revenue','discount_applied','net_revenue']].head(10)

,transaction_id,quantity,gross_revenue,discount_applied,net_revenue
63771,63772,4,1858.32,0.00,1858.32
89725,89726,4,1858.32,0.00,1858.32
39714,39715,4,1597.44,0.00,1597.44
57523,57524,3,1393.74,0.00,1393.74
53552,53553,3,1393.74,0.00,1393.74
8709,8710,4,1579.57,0.15,1342.63
58001,58002,4,1224.76,0.00,1224.76
15135,15136,4,1207.32,0.00,1207.32
95975,95976,4,1207.32,0.00,1207.32
97179,97180,4,1207.32,0.00,1207.32


In [29]:
# save clean data
df_transactions.to_csv(r'C:\protofolio\ecommerce_analytic\data bersih\fact_transaction_clean.csv', index=False)
print("File sudah tersimpan!")

File sudah tersimpan!
